# CS604 Data: One-Time Download to Google Drive

Download **all files** in the course dataset repo on Hugging Face to your Google Drive (by default in `My Drive/cs604/data/`). Right now this includes the train/val archives and image lists, `README.md` and `metadata.json`. After that, the training notebook extracts data straight from Drive, so you don't have to download from HF every time.

**How to use**
1. **A CPU runtime is enough** (`Runtime → Change runtime type → CPU`). No GPU is needed, so no compute units are wasted.
2. `Runtime → Run all`, and allow Google Drive access when asked.
3. **Keep this page open** until the last cell prints `Done`. It takes about 20–60 minutes, depending on network speed.

**Note**
- You need about **45GB** of free Drive space (`train.tar.gz` 40.1GB + `val.tar.gz` 4.4GB). Free Drive has only 15GB, so upgrade your storage first.
- Data is written to Drive while it downloads from HF; no extra copy is kept on the local Colab disk.
- If you get disconnected, just "Run all" again: finished files are skipped, and unfinished ones resume where they stopped.
- After the Phase 2 test set is released, just "Run all" once more: new files in the repo (like `test.tar.gz` and `test.txt`) are added, and files already downloaded are skipped.

In [ ]:
HF_REPO = "doem1997/cs604-course-data"
DRIVE_DATA_DIR = "/content/drive/MyDrive/cs604/data"  # the training notebooks read from here by default
SKIP_SPACE_CHECK = False     # set to True if Drive has enough space but the check is wrong

In [ ]:
import os, time, shutil
import requests
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_url
from huggingface_hub.hf_api import RepoFile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

## Check the files to download and the Drive space

In [ ]:
# List all files in the repo and download them from smallest to largest (train.tar.gz last)
repo_files = [f for f in HfApi().list_repo_tree(HF_REPO, repo_type="dataset", recursive=True) if isinstance(f, RepoFile)]
sizes = {f.path: f.size for f in repo_files}
names = sorted(sizes, key=sizes.get)


def drive_path(name):
    return os.path.join(DRIVE_DATA_DIR, name)


def in_drive(name):
    return os.path.exists(drive_path(name)) and os.path.getsize(drive_path(name)) == sizes[name]


def partial_bytes(name):
    part = drive_path(name) + ".part"
    return os.path.getsize(part) if os.path.exists(part) else 0


todo = [n for n in names if not in_drive(n)]
need = sum(sizes[n] - partial_bytes(n) for n in todo)
free = shutil.disk_usage(DRIVE_DATA_DIR).free
for n in names:
    status = "already in Drive" if n not in todo else f"to download ({partial_bytes(n) / 1e9:.1f} GB done)"
    print(f"{n:16s} {sizes[n] / 1e9:7.2f} GB   {status}")
print(f"\nNeed {need / 1e9:.1f} GB more; Drive reports {free / 1e9:.1f} GB free")
if need > free and not SKIP_SPACE_CHECK:
    raise RuntimeError("Not enough free space on Drive: free up space or upgrade your Google Drive storage."
                       " If you are sure there is enough space and the check is wrong, set SKIP_SPACE_CHECK = True and run again")

## Download to Drive (resumable)

In [ ]:
def download(name, retries=10):
    """Stream a file from HF into a .part file on Drive. If interrupted, resume with an HTTP Range request. Rename when done"""
    size, part = sizes[name], drive_path(name) + ".part"
    os.makedirs(os.path.dirname(part), exist_ok=True)
    url = hf_hub_url(HF_REPO, name, repo_type="dataset")
    for attempt in range(1, retries + 1):
        done = partial_bytes(name)
        if done > size:  # leftover file is broken, download from the start
            os.remove(part)
            done = 0
        if done < size:
            headers = {"Accept-Encoding": "identity"}
            if done:
                headers["Range"] = f"bytes={done}-"
            try:
                with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    if done and r.status_code != 206:  # server ignored the Range request, so start over
                        done = 0
                    with open(part, "ab" if done else "wb") as f, \
                            tqdm(total=size, initial=done, unit="B", unit_scale=True, desc=name) as bar:
                        for chunk in r.iter_content(chunk_size=8 << 20):
                            f.write(chunk)
                            bar.update(len(chunk))
            except (requests.RequestException, OSError) as e:
                print(f"{name}: attempt {attempt} interrupted ({e!r}); resuming in 10 s")
                time.sleep(10)
                if isinstance(e, OSError) and not os.path.isdir(DRIVE_DATA_DIR):
                    drive.mount("/content/drive", force_remount=True)  # remount Drive if the mount dropped
                continue
        if partial_bytes(name) == size:
            os.replace(part, drive_path(name))
            return
    raise RuntimeError(f"{name}: still incomplete after {retries} attempts; re-run this cell to resume")


t0 = time.time()
for name in todo:
    download(name)
print(f"All files written to {DRIVE_DATA_DIR} in {(time.time() - t0) / 60:.1f} min")

## Wait for the Drive upload to finish

Data written to `/content/drive` is first cached on the local Colab disk and then uploaded in the background. The next step waits until all data has really reached Google Drive. **Do not close the page or disconnect the runtime before it prints `Done`**.

In [ ]:
bad = [n for n in names if not in_drive(n)]
assert not bad, f"incomplete files: {bad}; re-run the download cell"
print("Waiting for Google Drive to finish uploading (can take a while for 40+ GB) ...")
t0 = time.time()
drive.flush_and_unmount()
print(f"Done in {(time.time() - t0) / 60:.1f} min. Files in Google Drive: {DRIVE_DATA_DIR}")
for n in names:
    print(f"  {n:16s} {sizes[n] / 1e9:7.2f} GB")

## Next steps

The `DRIVE_DATA_DIR` of the training notebook (`cs604_vit_adaptation.ipynb`) defaults to `/content/drive/MyDrive/cs604/data`. If the data is there, training extracts it straight from Drive to the local Colab disk and does not download from HF.

Each new session still needs to extract the data once (a few minutes), because training must read images from the local Colab disk: reading 450,000 small images one by one from Drive would be very slow.